In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal

# Define Pydantic models for scene analysis
class PersonAnalysis(BaseModel):
    positionInFrame: str = Field(description="Approximate location in the image frame (e.g., foreground, background)")
    position: str = Field(description="Body posture of the person (e.g., lying, sitting, standing)")
    estimatedAge: int = Field(ge=0, description="Estimated age of the person")
    activity: str = Field(description="What the person is doing (e.g., injured, helping, observing)")
    gender: Literal["male", "female", "non-binary", "other", "prefer not to say"]
    injurySeverity: Literal["critical", "moderate", "minor", "none"]
    conscious: bool = Field(description="Is the person conscious?")

class SceneAnalysis(BaseModel):
    numberOfPeople: int = Field(ge=0, description="The total number of people visible in the scene")
    people: List[PersonAnalysis]

# Prompts for analysis
sysprompt = "You are a careful and experienced rescuer. Answer precisely following the schema."
prompt = "Describe in detail the scene. Check out for humans, do not miss any."

# Path to your image
image_path = "images/accident_scene.png"


In [ ]:
from PIL import Image
im = Image.open(image_path)
im

In [ ]:
import base64
import json
import os
from openai import OpenAI
from IPython.display import Audio
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Initialize OpenAI client
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

# Encode image to base64
base64_image = encode_image(image_path)

# Extract structured scene analysis using OpenAI
gpt_response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": [{"type": "input_text", "text": sysprompt}]
        },
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}"
                }
            ]
        }
    ],
    text_format=SceneAnalysis
)

# Parse and validate with Pydantic
detailed_description = gpt_response.output_parsed

print("GPT Scene Analysis:")
print(detailed_description.model_dump_json(indent=2))

# Generate audio response with scene analysis
audio_response = openai_client.chat.completions.create(
    model="gpt-4o-audio-preview",
    modalities=["text", "audio"],
    audio={"voice": "alloy", "format": "wav"},
    messages=[
        {
            "role": "system",
            "content": "You are an experienced rescuer. Give your answer quickly and concisely. No time to lose."
        },
        {
            "role": "user",
            "content": (
                f"Consider the following detailed description of the accident scene: "
                f"{detailed_description.model_dump_json()}. Who should I help first?"
            )
        }
    ]
)

print("\nAudio Response:")
print(audio_response.choices[0].message)

# Extract and play audio
wav_bytes = base64.b64decode(audio_response.choices[0].message.audio.data)
Audio(wav_bytes, autoplay=True)


In [ ]:
detailed_description

GEMINI

In [ ]:
from dotenv import load_dotenv
from google import genai
from PIL import Image
import json
from openai import OpenAI
import base64
from IPython.display import Audio

# Load environment variables
load_dotenv()


In [ ]:
# Initialize Gemini client
gemini_client = genai.Client()

# Load image
im = Image.open(image_path)

# Generate JSON schema from Pydantic model
schema = SceneAnalysis.model_json_schema()

config = {
    "response_mime_type": "application/json",
    "response_json_schema": schema
}

# Extract structured scene analysis using Gemini
gemini_response = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        im,
        (
            f"Context: {sysprompt}. "
            "Follow this JSON schema exactly for the response. "
            f"Input: {prompt}"
        )
    ],
    config=config
)

# Parse and validate with Pydantic
gemini_data = json.loads(gemini_response.text)
detailed_response_gemini = SceneAnalysis(**gemini_data)

print("Gemini Scene Analysis:")
print(detailed_response_gemini.model_dump_json(indent=2))


In [ ]:
print(detailed_response_gemini)

In [ ]:
# Generate audio response from Gemini analysis using OpenAI
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

audio_response_gemini = openai_client.chat.completions.create(
    model="gpt-4o-audio-preview",
    modalities=["text", "audio"],
    audio={"voice": "alloy", "format": "wav"},
    messages=[
        {
            "role": "system",
            "content": "You are an experienced rescuer. Give your answer quickly and concisely. No time to lose."
        },
        {
            "role": "user",
            "content": (
                f"Consider the following detailed description of the accident scene: "
                f"{detailed_response_gemini.model_dump_json()}. Who should I help first?"
            )
        }
    ]
)

print("Audio Response from Gemini Analysis:")
print(audio_response_gemini.choices[0].message)

# Extract and play audio
wav_bytes = base64.b64decode(audio_response_gemini.choices[0].message.audio.data)
Audio(wav_bytes, autoplay=True)
